# Lab 05: RAG with Bedrock Knowledge Bases

## Overview
In this lab you will build an end-to-end Retrieval-Augmented Generation (RAG) pipeline using Amazon Bedrock Knowledge Bases. You will create a Knowledge Base connected to an S3 data source, ingest documents, and query with both the RetrieveAndGenerate and Retrieve APIs. You will also compare this managed RAG approach with the custom vector search pipeline from Lab 04.

## Learning Objectives
- Create a Bedrock Knowledge Base with an OpenSearch Serverless vector store
- Configure an S3 data source and run document ingestion
- Use the RetrieveAndGenerate API for fully managed RAG
- Use the Retrieve API for custom generation workflows
- Understand metadata filtering and retrieval configuration
- Compare managed RAG vs custom RAG approaches

## Exam Domain
**Building Generative AI Applications (30%)** — This lab covers Task 2.1 (RAG solutions) focusing on the managed Knowledge Bases service: creating knowledge bases, configuring data sources, ingestion pipelines, and the RetrieveAndGenerate vs Retrieve APIs.

## Architecture Diagram
![Lab 05 Architecture](../assets/diagrams/lab-05-rag-knowledge-bases.png)

```mermaid
flowchart LR
    S3["S3 Bucket\n(Documents)"] --> DS["Data Source\nConnector"]
    DS --> Ingest["Ingestion Job\n(Chunk + Embed)"]
    Ingest --> VS["OpenSearch\nServerless\n(Vector Store)"]
    VS --> Retrieve["Retrieve API\n(Raw Chunks)"]
    VS --> RAG["RetrieveAndGenerate\n(Managed RAG)"]
    RAG --> Answer["Answer +\nCitations"]
    Retrieve --> Custom["Custom Prompt\n+ Generation"]
    Custom --> Answer
```

In [ ]:
%pip install boto3 langchain-aws langchain --quiet

In [ ]:
import boto3, json, os, time

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

# bedrock-agent for control plane (create/manage KBs), bedrock-agent-runtime for data plane (query)
bedrock_agent = session.client("bedrock-agent")
bedrock_agent_runtime = session.client("bedrock-agent-runtime")
bedrock_runtime = session.client("bedrock-runtime")
s3 = session.client("s3")
sts = session.client("sts")
aoss = session.client("opensearchserverless")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"

# Exam note: Knowledge Bases supports OpenSearch Serverless, Aurora pgvector, Pinecone, Redis
collections = aoss.batch_get_collection(names=["genai-lab-vectors"])
COLLECTION_ARN = collections["collectionDetails"][0]["arn"]
COLLECTION_ENDPOINT = collections["collectionDetails"][0]["collectionEndpoint"]
COLLECTION_HOST = COLLECTION_ENDPOINT.replace("https://", "")

print(f"Account:        {ACCOUNT_ID}")
print(f"S3 bucket:      {BUCKET}")
print(f"Bedrock role:   {BEDROCK_ROLE_ARN}")
print(f"Collection ARN: {COLLECTION_ARN}")
print(f"Collection host: {COLLECTION_HOST}")

## A. Create a Knowledge Base

A **Bedrock Knowledge Base** is a managed RAG service that automates chunking, embedding, storage, and retrieval -- you declare the configuration, Bedrock orchestrates the pipeline. This replaces the manual approach from Lab 04.

Three pieces of configuration are required:

- **Knowledge Base config** -- embedding model (Titan Embeddings V2) and type (`VECTOR`)
- **Storage config** -- where to store vectors (our OpenSearch Serverless collection)
- **IAM role** -- grants Bedrock permission to read S3, invoke the embedding model, and write to the vector store

In [ ]:
# Pre-create the kb-vectors index in OpenSearch Serverless
# KB requires the index to exist BEFORE creation — Bedrock won't auto-create for OSS
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

credentials = session.get_credentials().get_frozen_credentials()
# OSS uses "aoss" service name (not "es") for SigV4 signing
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    REGION,
    "aoss",
    session_token=credentials.token,
)

oss_client = OpenSearch(
    hosts=[{"host": COLLECTION_HOST, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
)

KB_INDEX_NAME = "kb-vectors"

# Idempotent — delete if exists from a previous run
if oss_client.indices.exists(index=KB_INDEX_NAME):
    oss_client.indices.delete(index=KB_INDEX_NAME)
    print(f"Deleted existing index: {KB_INDEX_NAME}")

# dimension=1024 matches Titan Embeddings V2 output; FAISS HNSW gives best recall/speed tradeoff
index_body = {
    "settings": {
        "index": {
            "knn": True,
        }
    },
    "mappings": {
        "properties": {
            "embedding": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "engine": "faiss",
                    "name": "hnsw",
                },
            },
            "text": {"type": "text"},
            "metadata": {"type": "text"},
        }
    },
}

oss_client.indices.create(index=KB_INDEX_NAME, body=index_body)
print(f"Created index: {KB_INDEX_NAME}")

# OSS needs time to propagate the new index before it can accept documents
print("Waiting for index to become ready...")
time.sleep(30)
print("Index ready.")

In [ ]:
# Create Knowledge Base (or reuse existing one)
try:
    kb = bedrock_agent.create_knowledge_base(
        name="genai-lab-knowledge-base",
        roleArn=BEDROCK_ROLE_ARN,
        knowledgeBaseConfiguration={
            "type": "VECTOR",  # Exam: VECTOR vs KENDRA (structured) vs GRAPH — know when to use each
            "vectorKnowledgeBaseConfiguration": {
                "embeddingModelArn": f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2:0"
            }
        },
        storageConfiguration={
            "type": "OPENSEARCH_SERVERLESS",
            "opensearchServerlessConfiguration": {
                "collectionArn": COLLECTION_ARN,
                "vectorIndexName": "kb-vectors",
                # fieldMapping ties Bedrock's internal fields to your OSS index schema
                "fieldMapping": {
                    "vectorField": "embedding",
                    "textField": "text",
                    "metadataField": "metadata"
                }
            }
        }
    )
    KB_ID = kb["knowledgeBase"]["knowledgeBaseId"]
    print(f"Knowledge Base created: {KB_ID}")
except bedrock_agent.exceptions.ConflictException:
    # KB already exists from a previous run — find and reuse it
    kbs = bedrock_agent.list_knowledge_bases()
    KB_ID = next(
        kb["knowledgeBaseId"] for kb in kbs["knowledgeBaseSummaries"]
        if kb["name"] == "genai-lab-knowledge-base"
    )
    print(f"Knowledge Base already exists, reusing: {KB_ID}")

print(f"KB_ID = {KB_ID}")

# KB creation is async — must wait for ACTIVE before attaching data sources
print("Waiting for Knowledge Base to become ACTIVE...")
while True:
    kb_status = bedrock_agent.get_knowledge_base(knowledgeBaseId=KB_ID)
    status = kb_status["knowledgeBase"]["status"]
    print(f"  Status: {status}")
    if status == "ACTIVE":
        break
    elif status in ("FAILED", "DELETE_IN_PROGRESS"):
        raise Exception(f"Knowledge Base creation failed: {status}")
    time.sleep(10)
print("Knowledge Base is ACTIVE.")

### Understanding the Configuration

| Parameter | Purpose |
|-----------|---------|
| `type: VECTOR` | Tells Bedrock this is a vector-based knowledge base (as opposed to structured/graph) |
| `embeddingModelArn` | The foundation model used to convert text chunks into vector embeddings |
| `collectionArn` | The OpenSearch Serverless collection that stores the vectors |
| `vectorIndexName` | The index within the collection where vectors are written (`kb-vectors`) |
| `vectorField` / `textField` / `metadataField` | Maps Bedrock's internal fields to the OpenSearch index field names |

The `fieldMapping` tells Bedrock which OpenSearch fields to use for storing each piece of data. Bedrock creates the index automatically if it does not already exist.

## B. Configure Data Source and Ingest Documents

A **data source** tells the Knowledge Base where to find documents. Bedrock supports **S3, Confluence, SharePoint, Salesforce, and web crawlers**.

When you start an **ingestion job**, Bedrock automatically:
- Reads documents from the data source
- Splits them into chunks (~300 tokens, 20% overlap by default)
- Generates embeddings via the configured model
- Stores vectors + metadata in the vector store

This replaces the manual chunking, embedding, and indexing code from Lab 04.

In [ ]:
# Create S3 data source pointing to the whitepapers folder
# inclusionPrefixes scopes ingestion to a specific S3 path — avoids indexing unrelated files
ds = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID,
    name="aws-whitepapers",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{BUCKET}",
            "inclusionPrefixes": ["whitepapers/"]
        }
    }
)
DS_ID = ds["dataSource"]["dataSourceId"]
print(f"Data source created: {DS_ID}")
print(f"Status: {ds['dataSource']['status']}")

### Start Ingestion

Ingestion is **asynchronous**. We start the job and poll until complete (typically 1-5 minutes depending on document count/size). Re-running ingestion is **incremental** -- only new/modified documents are processed.

In [ ]:
# Start ingestion job
job = bedrock_agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
JOB_ID = job["ingestionJob"]["ingestionJobId"]
print(f"Ingestion job started: {JOB_ID}")

# Poll until complete
while True:
    status = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DS_ID,
        ingestionJobId=JOB_ID
    )
    job_status = status["ingestionJob"]["status"]
    print(f"  Status: {job_status}")

    if job_status in ("COMPLETE", "FAILED", "STOPPED"):
        break
    time.sleep(15)

# Show ingestion statistics
stats = status["ingestionJob"].get("statistics", {})
print(f"\nIngestion complete!")
print(f"  Documents scanned:  {stats.get('numberOfDocumentsScanned', 'N/A')}")
print(f"  Documents indexed:  {stats.get('numberOfNewDocumentsIndexed', 0) + stats.get('numberOfModifiedDocumentsIndexed', 0)}")
print(f"  Documents failed:   {stats.get('numberOfDocumentsFailed', 'N/A')}")

## C. RetrieveAndGenerate -- Managed RAG

**RetrieveAndGenerate** is Bedrock's fully managed RAG endpoint. One API call does everything:

1. Embeds your query
2. Performs vector search
3. Constructs a prompt with retrieved context
4. Sends it to the specified foundation model
5. Returns the answer **with source citations**

This is the simplest path to RAG -- no prompt management, no retrieval logic, no context window math.

In [ ]:
# RetrieveAndGenerate handles retrieval + prompt + generation in one call
# Exam: this is the simplest RAG API — no custom prompt needed
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": "What are the six pillars of the AWS Well-Architected Framework?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            # modelArn specifies which FM generates the answer from retrieved context
            "modelArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:inference-profile/us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        }
    }
)

print("Answer:")
print(response["output"]["text"])
print("\nSources:")
for citation in response.get("citations", []):
    for ref in citation.get("retrievedReferences", []):
        loc = ref.get("location", {}).get("s3Location", {})
        print(f"  - {loc.get('uri', 'unknown')}")

In [ ]:
# Query 2: Amazon Bedrock capabilities
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": "What is Amazon Bedrock and what are its key capabilities?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:inference-profile/us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        }
    }
)

print("Answer:")
print(response["output"]["text"])
print("\nSources:")
for citation in response.get("citations", []):
    for ref in citation.get("retrievedReferences", []):
        loc = ref.get("location", {}).get("s3Location", {})
        print(f"  - {loc.get('uri', 'unknown')}")

In [ ]:
# Query 3: Security best practices
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": "What are the best practices for securing generative AI applications on AWS?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:inference-profile/us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        }
    }
)

print("Answer:")
print(response["output"]["text"])
print("\nSources:")
for citation in response.get("citations", []):
    for ref in citation.get("retrievedReferences", []):
        loc = ref.get("location", {}).get("s3Location", {})
        print(f"  - {loc.get('uri', 'unknown')}")

## D. Retrieve Only -- Custom Generation

The **Retrieve** API returns raw document chunks without generating a response, giving you full control over the generation step.

**Exam: RetrieveAndGenerate vs Retrieve -- when to use each:**

| Use Case | API |
|----------|-----|
| Simple Q&A with citations | **RetrieveAndGenerate** |
| Custom prompt templates | **Retrieve** |
| Multi-step reasoning pipelines | **Retrieve** |
| Different retrieval and generation models | **Retrieve** |
| Pre/post-processing of chunks | **Retrieve** |
| Maximum simplicity | **RetrieveAndGenerate** |

### Step 1: Retrieve Relevant Chunks

Call the Retrieve API to get the top 5 document chunks. Each result includes chunk text, a relevance score, and source metadata.

In [ ]:
# Retrieve API returns raw chunks — no generation, no prompt construction
results = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": "What is Amazon Bedrock?"},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            # numberOfResults=5 balances recall vs context window usage
            # Too many chunks waste tokens; too few risk missing relevant info
            "numberOfResults": 5
        }
    }
)

for i, result in enumerate(results["retrievalResults"]):
    score = result.get("score", 0)
    text = result["content"]["text"][:200]
    source = result.get("location", {}).get("s3Location", {}).get("uri", "unknown")
    print(f"Chunk {i+1} (score: {score:.4f})")
    print(f"  Source: {source}")
    print(f"  Text:   {text}...")
    print()

### Step 2: Build a Custom Prompt and Generate

Assemble retrieved chunks into a custom prompt. This gives full control over prompt template, format instructions, and persona -- things RetrieveAndGenerate does not support.

In [ ]:
# Build custom prompt — this is why you'd choose Retrieve over RetrieveAndGenerate:
# full control over prompt template, instructions, and output format
context = "\n\n".join([r["content"]["text"] for r in results["retrievalResults"]])

prompt = f"""Based on the following context from AWS documentation, answer the question.
If the context does not contain enough information, say so clearly.

Context:
{context}

Question: What is Amazon Bedrock and what are its key features?

Provide a concise answer in 3-4 sentences."""

# Using Converse API (not RetrieveAndGenerate) — we control the generation model separately
response = bedrock_runtime.converse(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{"role": "user", "content": [{"text": prompt}]}],
    inferenceConfig={"maxTokens": 512}
)

print("Custom RAG Answer:")
print(response["output"]["message"]["content"][0]["text"])

## E. Appendix -- LangChain Integration (Optional)

> **Optional.** LangChain's `AmazonKnowledgeBasesRetriever` wraps the same Retrieve API into its retriever abstraction. Useful if you're building LangChain chains/agents; the native Boto3 APIs above are sufficient for most use cases and the exam.

In [ ]:
# Optional: LangChain retriever wrapping Bedrock Knowledge Bases
from langchain_aws import AmazonKnowledgeBasesRetriever

retriever = AmazonKnowledgeBasesRetriever(
    knowledge_base_id=KB_ID,
    retrieval_config={"vectorSearchConfiguration": {"numberOfResults": 5}},
)

docs = retriever.invoke("What is the Well-Architected Framework?")

print(f"Retrieved {len(docs)} documents via LangChain:\n")
for i, doc in enumerate(docs):
    score = doc.metadata.get("score", "N/A")
    score_str = f"{score:.4f}" if isinstance(score, float) else str(score)
    print(f"Document {i+1} (score: {score_str})")
    print(f"  {doc.page_content[:200]}...")
    print()

In [ ]:
# Optional: LangChain RAG chain with Knowledge Base retriever
from langchain_aws import ChatBedrock
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatBedrock(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=REGION
)

template = """Answer the question based only on the following context.
If the context does not contain enough information, say so.

Context:
{context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

# Build the chain: retrieve -> format -> generate
chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke("How does Amazon Bedrock support responsible AI?")
print("LangChain RAG Answer:")
print(answer)

## Cleanup

Delete the Knowledge Base and its data source. **Do not** delete the OpenSearch Serverless collection itself — it is shared infrastructure used by other labs and managed by `setup-resources.py`.

In [ ]:
# Delete data source first, then the Knowledge Base
bedrock_agent.delete_data_source(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
print(f"Deleted data source: {DS_ID}")

bedrock_agent.delete_knowledge_base(knowledgeBaseId=KB_ID)
print(f"Deleted Knowledge Base: {KB_ID}")

print("\nKnowledge Base and data source deleted.")
print("OpenSearch Serverless collection 'genai-lab-vectors' is preserved for other labs.")

# Delete the kb-vectors index from OpenSearch
oss_client.indices.delete(index=KB_INDEX_NAME)
print(f"Deleted index: {KB_INDEX_NAME}")

## Key Takeaways

1. **Bedrock Knowledge Bases automate the RAG pipeline** — document ingestion, chunking, embedding, and vector storage are fully managed, replacing dozens of lines of custom code with a single configuration
2. **RetrieveAndGenerate is the fastest path to RAG** — one API call handles retrieval, prompt construction, and generation with automatic source citations
3. **Retrieve API enables custom workflows** — when you need custom prompts, multi-step reasoning, or different generation models, Retrieve gives you raw chunks to build your own pipeline
4. **Ingestion is incremental** — re-running ingestion only processes new or modified documents, making it efficient for production data sources that change over time
5. **Managed RAG trades flexibility for simplicity** — you cannot customize the retrieval prompt template in RetrieveAndGenerate, but you avoid managing embeddings, vector indices, and prompt engineering

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Knowledge Base | A managed Bedrock resource that connects a data source to a vector store and an embedding model, automating the entire RAG ingestion and retrieval pipeline |
| Data Source | A connector that tells the Knowledge Base where to find documents (S3, Confluence, SharePoint, web crawler); each Knowledge Base can have multiple data sources |
| Ingestion Job | An asynchronous process that reads documents from a data source, chunks them, generates embeddings, and stores the vectors in the configured vector store |
| RetrieveAndGenerate | A single API call that performs retrieval from the Knowledge Base and generation with a foundation model, returning an answer with source citations |
| Retrieve | An API that returns raw document chunks from the Knowledge Base without generation, enabling custom prompt construction and multi-step workflows |
| Vector Store | The backend database (OpenSearch Serverless, Pinecone, or Redis) that stores and searches the embedding vectors created during ingestion |
| Chunking Strategy | The method used to split documents into smaller segments before embedding; Bedrock supports fixed-size, default, semantic, and hierarchical chunking strategies |

## Exam Preparation — Common Exam Question Patterns

**Q: What is the difference between RetrieveAndGenerate and Retrieve APIs in Bedrock Knowledge Bases?**
A: RetrieveAndGenerate performs end-to-end RAG in a single call — it retrieves relevant chunks, constructs a prompt, sends it to a foundation model, and returns the generated answer with citations. Retrieve only returns the relevant document chunks, giving developers full control over prompt construction, model selection, and post-processing. Use RetrieveAndGenerate for simple Q&A and Retrieve when you need custom prompts or multi-step workflows.

**Q: What vector stores does Amazon Bedrock Knowledge Bases support?**
A: Bedrock Knowledge Bases supports Amazon OpenSearch Serverless, Amazon Aurora PostgreSQL (with pgvector), Pinecone, and Redis Enterprise Cloud as vector stores. OpenSearch Serverless is the most commonly used option because it is fully managed and scales automatically within AWS.

**Q: How does Bedrock Knowledge Bases handle document updates?**
A: When you re-run an ingestion job, Bedrock performs incremental ingestion — it identifies new and modified documents in the data source, processes only those documents, and updates the vector store accordingly. Deleted documents are also removed from the index. This makes it efficient for production data sources that change frequently.

**Q: What chunking strategies are available in Bedrock Knowledge Bases?**
A: Bedrock supports four chunking strategies: default (automatic ~300 tokens), fixed-size (configurable token count with overlap), semantic (groups related sentences), and hierarchical (parent-child chunks for better context). The choice depends on your document structure — semantic chunking works well for narrative documents, while fixed-size is predictable and efficient for uniform content.

**Q: When should you use Bedrock Knowledge Bases vs building a custom RAG pipeline?**
A: Use Knowledge Bases when you want a fully managed RAG solution with minimal code, automatic document syncing, and built-in chunking and embedding. Build a custom pipeline when you need custom chunking logic, specialized embedding models not available in Bedrock, complex pre-processing steps, or fine-grained control over the retrieval and generation stages. Knowledge Bases reduce operational overhead but offer less flexibility than custom implementations.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Titan Embeddings V2 — Ingestion (document chunking + embedding) | ~50-100 chunks, ~25K tokens | < $0.01 |
| Claude Sonnet 4.5 — Section C (RetrieveAndGenerate, 3 queries) | ~3 API calls, ~6K tokens | ~$0.10 |
| Claude Sonnet 4.5 — Section D (custom generation) | ~1 API call, ~2K tokens | ~$0.03 |
| Claude Sonnet 4.5 — Section E (LangChain chain, optional) | ~1 API call, ~2K tokens | ~$0.03 |
| OpenSearch Serverless — OCU time | ~2 OCUs x 1.25 hours | ~$2-3 |
| **Total** | | **~$2-3** |

The dominant cost is OpenSearch Serverless compute (OCU) time, billed per OCU-hour while the collection is active. The collection was created by `setup-resources.py` and will continue accruing charges until deleted via `cleanup-all.py`. Bedrock API costs for this lab are minimal.